# 5장 실습 ③ — 깊이의 벽

**PyTorch 판**

층을 깊게 쌓으면 무슨 일이 생기는지 봅니다.
본문 §5.4의 세 표를 이 노트북이 만듭니다.

> 이 노트북은 **실험 설계에 대한 교훈**도 함께 담고 있습니다.
> 실험 ①만 보고 결론을 내리면 틀립니다. ②와 ③까지 보십시오.

## 5.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 5.1 실험대 — 소용돌이

이 문제의 **바닥**을 먼저 재 둡니다. 직선 하나로 최대 얼마나 맞히는가.
앞으로 나오는 숫자가 이 값 근처면 **"사실상 직선 하나"**라는 뜻입니다.

In [ ]:
# 소용돌이 — 하이퍼파라미터가 결과를 실제로 바꾸는 문제.
# 사과 데이터는 무엇을 해도 0.95가 나와서 이 장의 실험대가 될 수 없다.
x, y = data.spirals(n=1600, seed=42)
s = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(s.summary())

plot.scatter2d(s.x_train, s.y_train, class_names=("무리 0", "무리 1"),
               xlabel="$x_1$", ylabel="$x_2$", title="소용돌이 — 이 장의 실험대")
plt.show()

# 이 문제의 바닥: 직선 하나로 최대 얼마나 맞히는가
rng = np.random.default_rng(0)
best = 0.0
for _ in range(4000):
    w, b = rng.normal(size=2), rng.normal() * 3
    a = metrics.accuracy(y, (x @ w + b > 0).astype(int))
    best = max(best, a, 1 - a)
dlbook.record("ch05_spiral_single_line_acc", best)
print("→ 0.7 근처의 결과가 나오면 '사실상 직선 하나'라는 뜻입니다.")

## 5.2 학습 함수 — 여기만 판마다 다릅니다

아래 셀 하나가 이 판의 방식으로 모델을 만들고 학습시킵니다.
**이 아래의 모든 셀은 세 판이 같습니다.**

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

dlbook.set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

_ACT = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

def _init_(m, how):
    if not isinstance(m, nn.Linear):
        return
    if how == "zeros":
        nn.init.zeros_(m.weight)
    elif how == "random_normal":
        nn.init.normal_(m.weight, 0.0, 0.05)
    elif how == "he_normal":
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
    else:                                    # glorot_uniform = Xavier
        nn.init.xavier_uniform_(m.weight)
    nn.init.zeros_(m.bias)

def train_model(depth=3, units=32, act="relu", init="glorot_uniform",
                lr=0.01, opt="adam", epochs=60, bs=32, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, history)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    layers_ = []
    prev = 2
    for _ in range(depth):
        layers_ += [nn.Linear(prev, units), _ACT[act]()]
        prev = units
    layers_ += [nn.Linear(prev, 1)]          # 시그모이드는 손실 함수 안에 있다
    model = nn.Sequential(*layers_).to(device)
    model.apply(lambda m: _init_(m, init))

    criterion = nn.BCEWithLogitsLoss()
    optimizer = {"adam": torch.optim.Adam,
                 "sgd": torch.optim.SGD,
                 "rmsprop": torch.optim.RMSprop}[opt](model.parameters(), lr=lr)

    ds = TensorDataset(torch.tensor(s.x_train, dtype=torch.float32),
                       torch.tensor(s.y_train, dtype=torch.float32).view(-1, 1))
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    xv = torch.tensor(s.x_val, dtype=torch.float32).to(device)
    yv = torch.tensor(s.y_val, dtype=torch.float32).view(-1, 1).to(device)

    history = {"loss": [], "val_loss": []}
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        losses = []
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            history["val_loss"].append(criterion(model(xv), yv).item())
        history["loss"].append(float(np.mean(losses)))

    model.eval()
    with torch.no_grad():
        logit = model(torch.tensor(s.x_test, dtype=torch.float32).to(device))
    pred = (logit.cpu().numpy().reshape(-1) > 0).astype("int64")
    return metrics.accuracy(s.y_test, pred), history

## 5.4 깊이의 벽 — 실험 ①

**층을 늘리면 성능이 좋아질까요.** 학습률을 0.01로 **고정**하고 깊이만 바꿉니다.

⚠ 20층까지 돌리면 시간이 꽤 걸립니다. 스모크 모드에서는 10층까지만 합니다.

In [ ]:
# 실험 ① — 깊이 × 활성화 함수. 학습률은 0.01로 고정한다.
# ⚠ 이 표만 보고 결론을 내리면 틀립니다. 실험 ②를 반드시 함께 보십시오.
depths = [2, 5, 10] if dlbook.smoke.is_smoke() else [2, 5, 10, 20]
acts = ["sigmoid", "tanh", "relu"]

print(f"{'층 수':<7}" + "".join(f"{a:>11}" for a in acts))
for d in depths:
    row = []
    for a in acts:
        acc, _ = train_model(depth=d, act=a, lr=0.01)
        row.append(acc)
        dlbook.record(f"ch05_depth{d}_{a}_acc", acc)
    print(f"{d:<7}" + "".join(f"{v:>11.3f}" for v in row))

## 5.5 실험 ② — 학습률까지 함께 바꾼다

실험 ①은 학습률을 고정해 놓고 깊이만 바꿨습니다. 그러면 관찰된 실패가
**깊이 탓인지 학습률 탓인지 갈라낼 수 없습니다.**

확인해 봅니다. **이 표가 실험 ①의 해석을 뒤집습니다.**

In [ ]:
# 실험 ② — 학습률까지 함께 바꾼다. **이 표가 실험 ①의 해석을 뒤집는다.**
# 은닉층 10개 고정. 활성화 함수와 학습률을 둘 다 바꾼다.
lrs = [0.001, 0.01] if dlbook.smoke.is_smoke() else [0.0003, 0.001, 0.003, 0.01, 0.03]

print(f"{'활성화':<10}" + "".join(f"{'lr='+str(l):>11}" for l in lrs))
for act in ("sigmoid", "tanh", "relu"):
    row = []
    for lr in lrs:
        acc, _ = train_model(depth=10, act=act, lr=lr)
        row.append(acc)
        dlbook.record(f"ch05_d10_{act}_lr{lr}_acc", acc)
    print(f"{act:<10}" + "".join(f"{v:>11.3f}" for v in row))

print()
print("→ 시그모이드는 **어떤 학습률로도** 0.72를 못 넘습니다. 이것이 기울기 소실입니다.")
print("→ tanh와 ReLU는 학습률만 낮추면 잘 됩니다. 실험 ①의 실패는 학습률 탓이었습니다.")

## 5.6 20층 ReLU는 왜 무너졌나

실험 ①에서 20층 ReLU가 0.500이었습니다. 깊이 탓일까요, 학습률 탓일까요.

In [ ]:
# 그러면 20층 ReLU가 무너진 것은 깊이 탓인가, 학습률 탓인가.
depths = [10, 20] if dlbook.smoke.is_smoke() else [10, 20, 30]
lrs2 = [0.001, 0.01]

print(f"{'층 수(ReLU)':<12}" + "".join(f"{'lr='+str(l):>11}" for l in lrs2))
for d in depths:
    row = []
    for lr in lrs2:
        acc, _ = train_model(depth=d, act="relu", lr=lr)
        row.append(acc)
        dlbook.record(f"ch05_relu_d{d}_lr{lr}_acc", acc)
    print(f"{d:<12}" + "".join(f"{v:>11.3f}" for v in row))

print()
print("→ ReLU에게 깊이의 벽은 없습니다. 학습률만 맞으면 30층도 됩니다.")
print("→ 대신 **깊어질수록 견딜 수 있는 학습률의 폭이 좁아집니다.**")

## 정리

- **시그모이드는 10층에서 어떤 학습률로도 안 됩니다.** 미분의 최댓값이 0.25라
  열 번 곱하면 백만분의 1이 됩니다. **이것이 기울기 소실입니다.**
- **ReLU에게 깊이의 벽은 없습니다.** 학습률만 맞으면 30층도 됩니다.
  대신 깊어질수록 견딜 수 있는 학습률의 폭이 좁아집니다.
- **한 번에 하나씩만 바꾸십시오.** 실험 ①만 보고 "깊이의 벽"이라고 결론
  내리면 틀립니다. 이 책의 초고가 실제로 그 함정에 빠졌습니다.

### 연습

1. 10층 시그모이드 모델에서 **각 층의 가중치 변화량**을 출력해,
   앞쪽 층이 거의 변하지 않았음을 보이십시오.
2. 10층 시그모이드에 He 초기화를 주면 나아집니까. 왜 그렇습니까.
3. 20층 ReLU 모델에 배치 정규화 층을 끼워 넣어 보십시오. (6장 선행 학습)
4. 실험 ①과 ②의 차이가 **실험 설계**에서 무엇을 말해 주는지 한 문단으로 쓰십시오.